# Interactive Generation — ConvVAE + CEM

- **Mode 1** — `z_seed` picks a fixed latent vector; digit sliders intervene on the two concepts while z stays the same.
- **Mode 2** — every slider move samples a **brand-new** z from N(0,1) and decodes it with the chosen digit concepts.

In [1]:
%matplotlib inline
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

from scripts.train_cem_vae import load_cem_vae

CHECKPOINT    = "../outputs/models/cem_vae_mnist"
CONCEPT_MEAN  = 5.0
CONCEPT_SCALE = 4.0

device = "cpu"
if torch.backends.mps.is_available():  device = "mps"
elif torch.cuda.is_available():         device = "cuda"

model, backbone_cfg, cem_cfg = load_cem_vae(CHECKPOINT, device=device)
print(f"Loaded on {device} | n_concepts={cem_cfg.n_concepts}, latent_dim={backbone_cfg.latent_dim}")

Loaded on mps | n_concepts=2, latent_dim=32


In [2]:
def generate(z, digit1, digit2):
    c = torch.tensor(
        [[(digit1 - CONCEPT_MEAN) / CONCEPT_SCALE,
          (digit2 - CONCEPT_MEAN) / CONCEPT_SCALE]],
        device=device
    )
    with torch.no_grad():
        emb, concept = model.cem(z, interventions=c)
        nomralised_concept = concept.cpu().numpy()
        real_concept = nomralised_concept * CONCEPT_SCALE + CONCEPT_MEAN
        print(f"Concepts: {real_concept}")
        recon  = model.decoder(emb).reconstruction
    img = recon[0].squeeze().cpu().float().numpy()
    return (img * 0.3081 + 0.1307).clip(0, 1)

## Mode 1 — Fixed z, move the concept sliders

Change **z seed** to pick a different latent vector from the prior.  
Then move **Digit 1** and **Digit 2** to intervene on the concepts while the same z is kept.

In [8]:
@interact(
    z_seed = IntSlider(min=0, max=99, value=0,  description='z seed:',  continuous_update=False),
    digit1 = IntSlider(min=0, max=10,  value=5,  description='Digit 1:', continuous_update=True),
    digit2 = IntSlider(min=0, max=10,  value=5,  description='Digit 2:', continuous_update=True),
)
def mode1(z_seed=0, digit1=5, digit2=5):
    torch.manual_seed(z_seed)
    z   = torch.randn(1, backbone_cfg.latent_dim, device=device)
    img = generate(z, digit1, digit2)
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"seed={z_seed}  |  digit1={digit1}, digit2={digit2}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=0, continuous_update=False, description='z seed:', max=99), IntSlider(va…

## Mode 2 — New z on every slider move

Every time you change a slider, a **fresh z** is sampled from N(0,1) and decoded with the digit values you set.

In [ ]:
@interact(
    digit1 = IntSlider(min=1, max=9, value=5, description='Digit 1:', continuous_update=True),
    digit2 = IntSlider(min=1, max=9, value=5, description='Digit 2:', continuous_update=True),
)
def mode2(digit1=5, digit2=5):
    z   = torch.randn(1, backbone_cfg.latent_dim, device=device)
    img = generate(z, digit1, digit2)
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.title(f"fresh z  |  digit1={digit1}, digit2={digit2}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=5, description='Digit 1:', max=9, min=1), IntSlider(value=5, description…

## Mode 3 — Optimise z to match target concepts

No intervention, no random z blending. We optimise z directly so that CEM predicts your target digits, then decode. Bypasses the pos/neg blending limitation entirely.

In [10]:
import torch.nn.functional as F

def optimise_z(digit1, digit2, n_steps=300, lr=0.05):
    c_target = torch.tensor(
        [[(digit1 - CONCEPT_MEAN) / CONCEPT_SCALE,
          (digit2 - CONCEPT_MEAN) / CONCEPT_SCALE]],
        device=device
    )
    z = torch.randn(1, backbone_cfg.latent_dim, device=device, requires_grad=True)
    opt = torch.optim.Adam([z], lr=lr)
    for _ in range(n_steps):
        _, concepts = model.cem(z)
        loss = F.mse_loss(concepts, c_target)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        emb, predicted = model.cem(z)
        recon = model.decoder(emb).reconstruction
    p = predicted[0].cpu().numpy() * CONCEPT_SCALE + CONCEPT_MEAN
    print(f"Target: [{digit1}, {digit2}]  Predicted after opt: [{p[0]:.1f}, {p[1]:.1f}]")
    img = recon[0].squeeze().cpu().float().numpy()
    return (img * 0.3081 + 0.1307).clip(0, 1)


In [11]:
@interact(
    digit1 = IntSlider(min=1, max=9, value=5, description="Digit 1:", continuous_update=False),
    digit2 = IntSlider(min=1, max=9, value=5, description="Digit 2:", continuous_update=False),
    n_steps = IntSlider(min=50, max=500, step=50, value=300, description="Steps:", continuous_update=False),
)
def mode3(digit1=5, digit2=5, n_steps=300):
    img = optimise_z(digit1, digit2, n_steps=n_steps)
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap="gray", vmin=0, vmax=1)
    plt.title(f"optimised z  |  digit1={digit1}, digit2={digit2}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


interactive(children=(IntSlider(value=5, continuous_update=False, description='Digit 1:', max=9, min=1), IntSl…